# 2 · Quem são os beneficiários dos rendimentos da dívida?

**Pergunta:** quando a União paga juros, para quem vai esse rendimento?

**Do que há evidência pública:**
- composição do estoque da DPMFi **por grupo de detentor** (Anexo 2.7 do RMD, série desde 2007);
- composição **por título** (Anexo 2.8) e juros apropriados da DPF (Anexo 2.9, conta I.2);
- estimativa do **fluxo de juros por grupo** (juros apropriados da DPF no ano × participação no estoque).

**O que é público:** o grupo detentor, o título e, em parte, o nível institucional — os fundos que divulgam carteira são nomeados pela CVM (CDA), os não-residentes aparecem por país de domicílio e as EFPCs têm o patrocínio nomeado pela PREVIC.

**O que NÃO é público:** o **cotista final** do fundo e a **carteira detalhada** dos fundos com dispensa de divulgação. O Requerimento 696/2019 (Câmara dos Deputados), que pedia a lista de credores ao STN, foi respondido com **negativa por sigilo bancário**. Qualquer "banqueiro X recebeu Y bilhões" publicado em rede social, sem fonte primária, é extrapolação.


In [1]:
import sys
import zipfile
from pathlib import Path
p = Path.cwd()
while not (p / "src").exists() and p != p.parent:
    p = p.parent
sys.path.insert(0, str(p))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def fmt_int(v):
    return f"{v:,}".replace(",", ".")


def fmt_num(v, dec=1):
    s = f"{v:,.{dec}f}"
    return s.replace(",", "X").replace(".", ",").replace("X", ".")


def fmt_bi(v):
    if abs(v) >= 1000:
        return f"R$ {fmt_num(v / 1000, 2)} tri"
    return f"R$ {fmt_int(round(v))} bi"


def fmt_pct(v, dec=1):
    return f"{fmt_num(v * 100, dec)}%"


pd.set_option("display.width", 170)
pd.set_option("display.float_format", lambda v: fmt_num(v, 1))

from src.viz.graficos import (
    estilo, salvar, AZUL, VERMELHO, VERDE, LARANJA, ROXO, CINZA, CORES_DETENTOR,
)
from src.processa import indicadores as ind

estilo()
z = zipfile.ZipFile(p / "dados" / "bruto" / "cvm" / "cda_fi_202610.zip")
print("diretório de dados:", p / "dados" / "processado")
print("CDA CVM:", z.filename.split("/")[-1])


diretório de dados: /home/vanessa/devel/juros_e_divida/dados/processado
CDA CVM: cda_fi_202610.zip


In [2]:
dz = ind.detentores_anuais()
part = dz.pivot_table(index="data", columns="detentor", values="participacao") * 100
part.tail(3).round(1)

detentor,Fundos de Investimento,Governo,Instituições Financeiras,Não-residentes,Outros,Previdência,Seguradoras,Tesouro Direto
data,,,,,,,,
2023-12-31,"23,5","3,8","29,7","9,5","4,5","23,0","4,0","2,0"
2024-12-31,"21,7","3,4","29,5","10,2","5,1","23,9","4,0","2,3"
2025-12-31,"20,8","2,9","32,9","10,3","4,3","22,8","3,5","2,6"


In [3]:
ordem = ["Instituições Financeiras", "Fundos de Investimento", "Previdência",
         "Não-residentes", "Tesouro Direto", "Seguradoras", "Governo", "Outros"]
fig, ax = plt.subplots(figsize=(11, 5.5))
ax.stackplot(part.index, [part[c].to_numpy() for c in ordem],
             labels=ordem, colors=[CORES_DETENTOR[c] for c in ordem], alpha=0.85)
ax.set_ylabel("% do estoque DPMFi")
ax.set_ylim(0, 100)
ax.set_title("Quem detém a dívida mobiliária federal em mercado (2007–2025)")
ax.legend(loc="upper center", ncol=4, fontsize=9, bbox_to_anchor=(0.5, -0.12))
salvar(fig, "02_detentores_area")


fig -> viz/figs/02_detentores_area.png


PosixPath('/home/vanessa/devel/juros_e_divida/viz/figs/02_detentores_area.png')

In [4]:
dez = dz[dz.data == dz.data.max()].sort_values("valor_bi", ascending=True)
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(dez.detentor, dez.valor_bi, color=[CORES_DETENTOR[c] for c in dez.detentor])
for i, (v, pr) in enumerate(zip(dez.valor_bi, dez.participacao)):
    ax.text(v + 30, i, f"{fmt_bi(v)} ({fmt_pct(pr)})", va="center", fontsize=9)
ax.set_xlabel("R$ bilhões (estoque em dezembro do último ano completo)")
ax.set_xlim(0, dez.valor_bi.max() * 1.32)
ax.set_title("Estoque da DPMFi por detentor (dezembro)")
salvar(fig, "02_detentores_estoque")


fig -> viz/figs/02_detentores_estoque.png


PosixPath('/home/vanessa/devel/juros_e_divida/viz/figs/02_detentores_estoque.png')

In [5]:
jf = ind.juros_por_detentor()
jf = jf[jf.ano < 2026]
conv = jf.pivot_table(index="ano", columns="detentor", values="juros_estimado_bi")
print("Juros apropriados da DPF por ano (RMD 2.9, R$ bi) — base da estimativa:")
fat = ind.fatores_anuais()
ref = fat[fat.ano < 2026].set_index("ano")["juros_apropriados_mi"] / 1000
print(ref.tail(12).round(0).to_string(float_format=lambda v: fmt_num(v, 0)))
print("\nComparação: juros nominais do setor público consolidado (SGS 4616):")
v = ind.juros_anuais()
sgs = v[v.ano < 2026].set_index("ano")["juros_mi"] / 1000
print(sgs.tail(12).round(0).to_string(float_format=lambda v: fmt_num(v, 0)))


Juros apropriados da DPF por ano (RMD 2.9, R$ bi) — base da estimativa:
ano
2014   243
2015   368
2016   330
2017   328
2018   343
2019   331
2020   353
2021   449
2022   556
2023   608
2024   762
2025   880

Comparação: juros nominais do setor público consolidado (SGS 4616):
ano
2014     311
2015     502
2016     407
2017     401
2018     379
2019     367
2020     312
2021     448
2022     586
2023     718
2024     950
2025   1.008


In [6]:
conv = conv.dropna(axis=1, how="all")
fig, ax = plt.subplots(figsize=(11, 5.5))
b = conv.loc[2014:]
ax.stackplot(b.index, [b[c].to_numpy() for c in b.columns],
             labels=list(b.columns),
             colors=[CORES_DETENTOR.get(c, "#999999") for c in b.columns], alpha=0.85)
ax.set_ylabel("R$ bilhões/ano (estimado)")
ax.set_title("Fluxo de juros estimado por grupo detentor (participação × juros apropriados da DPF)")
ax.legend(loc="upper center", ncol=4, fontsize=9, bbox_to_anchor=(0.5, -0.12))
salvar(fig, "02_juros_por_detentor")


fig -> viz/figs/02_juros_por_detentor.png


PosixPath('/home/vanessa/devel/juros_e_divida/viz/figs/02_juros_por_detentor.png')

**Premissa do gráfico acima:** participação do estoque em **dezembro** de cada
ano × **juros apropriados da DPF** no ano (RMD, anexo 2.9, conta I.2). Não é
dado oficial de fluxo por detentor — a União não publica o fluxo por credor; é
a **melhor aproximação possível com dados públicos**, boa para o tamanho
relativo dos beneficiários.

**Por que não o SGS 4616:** essa série mede os juros nominais do **setor público
consolidado** (União, estados, estatais e BCB) e não corresponde ao estoque da
DPMFi, que é só federal. Usar 4616 × participação da DPMFi mistura perímetros;
o anexo 2.9 (DPF) é o numerador compatível.


## Quem segura o quê: o risco por título (RMD 2.8)

O anexo 2.7 responde **quem** recebe; o 2.8 responde **qual risco cada grupo
carrega** — a mesma dívida tem títulos de remunerações diferentes (LFT =
Selic; NTN-B = IPCA + juro real; LTN/NTN-F = pré-fixados) e quem segura cada
um não é o mesmo. Anexo 2.8 do RMD, desde 2011.


In [7]:
dt = pd.read_parquet(p / "dados" / "processado" / "rmd_detentores_titulo.parquet")
rotulos = {"Inst. Financeiras": "Instituições Financeiras",
           "Fundos": "Fundos de Investimento",
           "Não-residente": "Não-residentes"}
dt["detentor"] = dt.detentor.replace(rotulos)
ult = dt[dt.data == dt.data.max()]
compos = ult.pivot_table(index="detentor", columns="titulo", values="participacao")
compos = compos.reindex(ordem)[["LFT", "LTN", "NTN-B", "NTN-F"]]
print(f"Participação por título, {ult.data.max():%m/%Y} (%):")
print((compos * 100).round(1).to_string(float_format=lambda v: fmt_num(v, 1)))

agg = ult.groupby("detentor").valor_bi.sum().reindex(ordem)
print(f"\nAgregado por grupo (RMD 2.8), {ult.data.max():%m/%Y} — R$ bi:")
print(agg.round(0).to_string(float_format=lambda v: fmt_num(v, 0)))
print("\nParticipação no estoque total (%):")
print((agg / agg.sum() * 100).round(1).to_string(float_format=lambda v: fmt_num(v, 1)))


Participação por título, 09/2026 (%):
titulo                    LFT  LTN  NTN-B  NTN-F
detentor                                        
Instituições Financeiras 37,4 43,6   11,6   30,2
Fundos de Investimento   32,3 10,1   15,5    6,0
Previdência              17,4  7,6   47,0    6,9
Não-residentes            4,2 26,4    1,9   40,2
Tesouro Direto            2,3  2,6    6,1    0,9
Seguradoras               1,9  1,7    6,1    3,7
Governo                   0,3  5,2    4,3    8,2
Outros                    4,2  2,8    7,5    3,8

Agregado por grupo (RMD 2.8), 09/2026 — R$ bi:
detentor
Instituições Financeiras   2.831
Fundos de Investimento     2.073
Previdência                2.025
Não-residentes               866
Tesouro Direto               279
Seguradoras                  295
Governo                      231
Outros                       438

Participação no estoque total (%):
detentor
Instituições Financeiras   31,3
Fundos de Investimento     22,9
Previdência                22,4
Não-reside

In [8]:
fig, ax = plt.subplots(figsize=(10, 5))
base = compos
bottom = None
for det in [c for c in ordem if c in base.index]:
    vals = base.loc[det] * 100
    ax.bar(base.columns, vals, bottom=bottom, label=det,
           color=CORES_DETENTOR.get(det, "#999999"), alpha=0.9)
    bottom = vals if bottom is None else bottom + vals
ax.set_ylabel("% do estoque do título")
ax.set_ylim(0, 100)
ax.set_title(f"Quem segura cada título da DPMFi (RMD 2.8, {ult.data.max():%m/%Y})")
ax.legend(fontsize=9, frameon=False, ncol=2)
salvar(fig, "02_detentores_por_titulo")


fig -> viz/figs/02_detentores_por_titulo.png


PosixPath('/home/vanessa/devel/juros_e_divida/viz/figs/02_detentores_por_titulo.png')

In [9]:
# fluxo de juros por detentor x titulo: estoque x custo medio 12m por titulo
cm = pd.read_parquet(p / "dados" / "processado" / "rmd_custo_medio.parquet")
c25 = cm[(cm.tabela == "estoque 12m") & (cm.data.dt.year == 2025)]
custo = c25.groupby("escopo").taxa_pct_aa.mean() / 100
e25 = dt[dt.data.dt.year == 2025].groupby(["titulo", "detentor"]).valor_bi.mean()
fluxo = {t: e25.xs(t) * custo.get(t, float("nan"))
         for t in ["LFT", "LTN", "NTN-B", "NTN-F"]}
tab = pd.DataFrame(fluxo).fillna(0)
tab["TOTAL"] = tab.sum(axis=1)
tab = tab.sort_values("TOTAL", ascending=False)
print("Fluxo de juros estimado 2025, por detentor e título (R$ bi):")
print(tab.round(0).to_string(float_format=lambda v: fmt_num(v, 0)))
oficial = ind.fatores_anuais().set_index("ano").loc[2025, "juros_apropriados_mi"] / 1000
print(f"\nchecagem: estimado {fmt_bi(tab.TOTAL.sum())} vs juros apropriados da DPF "
      f"em 2025 (RMD 2.9, I.2): {fmt_bi(oficial)}")
print("  (não é validação independente: o custo médio por título e o total vêm "
      "da mesma família de dados do RMD; serve para checar consistência interna)")


Fluxo de juros estimado 2025, por detentor e título (R$ bi):
                          LFT  LTN  NTN-B  NTN-F  TOTAL
detentor                                               
Instituições Financeiras  176   57     37     15    285
Previdência                88   12     97      2    200
Fundos de Investimento    154   11     32      2    200
Não-residentes             16   40      6     24     86
Outros                     23    3     12      1     39
Seguradoras                12    2     13      2     30
Governo                     1   10     10      5     25
Tesouro Direto              8    2     10      0     21

checagem: estimado R$ 886 bi vs juros apropriados da DPF em 2025 (RMD 2.9, I.2): R$ 880 bi
  (não é validação independente: o custo médio por título e o total vêm da mesma família de dados do RMD; serve para checar consistência interna)


**O que a fatiagem mostra (e o 2.7 não):**

- a **previdência** concentra as **NTN-Bs** (47% do título de inflação; 34% em
  2011) — e o seu passivo é indexado à inflação: é hedge estrutural, não aposta;
- os **não-residentes** concentram as **pré-fixadas** (26% das LTN, 40% das
  NTN-F) — a aposta estrangeira no juro nominal;
- **bancos e fundos** concentram as **LFTs** (69%) — a rolagem indexada à
  Selic, o título líquido de curto prazo;
- o **Tesouro Direto** (pessoas físicas) é marginal: 2,3% das LFTs, 6,1% das
  NTN-Bs, ~R$ 21 bi de juros/ano (2,4% do total).

As duas tabelas do RMD são consistentes **por construção** — o 2.8 soma o mesmo
estoque do 2.7, e o agregado por grupo reproduz as participações de set/2026
(31,3 / 22,9 / 22,4 / 9,6%). A checagem que acrescenta algo é a estimativa de
juros por título contra o total do 2.9: R$ 886 bi vs R$ 880 bi — próxima, mas
**não independente** (as duas usam a mesma família de dados do RMD).


## Quem, pelo nome: a carteira dos fundos (CVM CDA)

A CVM publica a carteira mensal dos fundos que **divulgam posição**, com o nome
(`DENOM_SOCIAL`) e a posição em títulos públicos. É o detalhe que o RMD 2.7
não dá: os **fundos que carregam a dívida, por nome** — para os fundos sem
dispensa de divulgação. Fonte: CDA, bloco BLC_1.


In [10]:
cvm = pd.read_parquet(p / "dados" / "processado" / "cvm_fundos_titulos.parquet")
tot = cvm.valor_bi.sum()
print(f"títulos públicos em fundos (CVM, {cvm.mes.max():%m/%Y}): {fmt_bi(tot)} "
      f"em {fmt_int(cvm.fundo.nunique())} classes de fundos")
top = cvm.groupby("fundo").valor_bi.sum().nlargest(12)
print("\nmaiores classes por posição em títulos públicos (R$ bi):")
for nome, v in top.items():
    print(f"  {fmt_num(v):>8}  {nome[:72]}")


títulos públicos em fundos (CVM, 10/2026): R$ 3,16 tri em 5.993 classes de fundos

maiores classes por posição em títulos públicos (R$ bi):
     274,4  BRASILPREV TOP TPF FUNDO DE INVESTIMENTO FINANCEIRO RENDA FIXA RESPONSAB
     153,9  BB RF IV FUNDO DE INVESTIMENTO FINANCEIRO RENDA FIXA LONGO PRAZO RESP LI
     105,3  CAIXA PÓS FIXADO PREVINVEST PREVIDENCIÁRIO FUNDO DE INVESTIMENTO FINANCE
      81,7  FUNDO DE INVESTIMENTO FINANCEIRO RENDA FIXA CFIN TÍTULOS PÚBLICOS PLENAS
      50,6  CAIXA GESTÃO PREVINVEST PREVIDENCIÁRIO FUNDO DE INVESTIMENTO FINANCEIRO 
      50,3  FOX SOBERANO II FUNDO DE INVESTIMENTO FINANCEIRO RENDA FIXA - RESPONSABI
      49,2  SPECIAL RENDA FIXA REFERENCIADO DI FUNDO DE INVESTIMENTO FINANCEIRO RESP
      42,1  BB EXTRAMERCADO EXCLUSIVO FAT FUNDO DE INVESTIMENTO FINANCEIRO RENDA FIX
      41,4  BB TOP RENDA FIXA SIMPLES FUNDO DE INVESTIMENTO FINANCEIRO RESPONSABILID
      35,0  ITAÚ VERSO A RENDA FIXA REFERENCIADO DI FUNDO DE INVESTIMENTO FINANCEIRO
      33,5

In [11]:
# por trás dos fundos: os grandes bancos e gestoras
import re
import unicodedata

def _grupo(n):
    u = unicodedata.normalize("NFKD", str(n)).encode("ascii", "ignore").decode().upper()
    if "BRASILPREV" in u:
        return "BRASILPREV"
    if re.search(r"\bBB\b", u) or "BANCO DO BRASIL" in u:
        return "BB"
    if "CAIXA" in u:
        return "CAIXA"
    if "ITAU" in u:
        return "ITAU"
    if "BRADESCO" in u:
        return "BRADESCO"
    if "SANTANDER" in u:
        return "SANTANDER"
    if "BTG" in u:
        return "BTG"
    if "XP" in u:
        return "XP"
    return "outros"

cvm["grupo"] = cvm.fundo.map(_grupo)
por_grupo = cvm.groupby("grupo").valor_bi.sum().sort_values(ascending=False)
print("posição em títulos públicos por banco/gestora:")
for grupo, v in por_grupo.items():
    print(f"  {grupo:12s} {fmt_bi(v)}")

# confronto com o RMD 2.7 (meses mais próximos disponíveis)
conf = pd.read_csv(z.open("cda_fi_CONFID_202610.csv"), sep=";", encoding="latin-1",
                   low_memory=False)
conf["VL"] = pd.to_numeric(conf.VL_MERC_POS_FINAL, errors="coerce") / 1e9
confid_bi = conf[conf.TP_APLIC == "Títulos Públicos"].VL.sum()
d27 = pd.read_parquet(p / "dados" / "processado" / "rmd_detentores.parquet")
mes_cvm = cvm.mes.max()
mes_rmd = d27.data.max()
d27m = d27[d27.data == mes_rmd]
fp = d27m[d27m.detentor.isin(["Fundos de Investimento", "Previdência"])].valor_bi.sum()
resid = fp - (tot + confid_bi)
print(f"\nCVM ({mes_cvm:%m/%Y}): {fmt_bi(tot)} divulgados + {fmt_bi(confid_bi)} "
      f"confidenciais = {fmt_bi(tot + confid_bi)}")
print(f"RMD 2.7 Fundos+Previdência ({mes_rmd:%m/%Y}): {fmt_bi(fp)} → resíduo {fmt_bi(resid)}")
print("  (o resíduo NÃO é 'a previdência fechada': mistura EFPCs, diferenças de")
print("   perímetro e classificação; a PREVIC mede os títulos federais das EFPCs")
print("   em R$ 921 bi — medida direta separada, não a diferença entre as fontes)")


posição em títulos públicos por banco/gestora:
  outros       R$ 983 bi
  BB           R$ 680 bi
  CAIXA        R$ 419 bi
  BRASILPREV   R$ 394 bi
  ITAU         R$ 313 bi
  BRADESCO     R$ 207 bi
  SANTANDER    R$ 148 bi
  BTG          R$ 7 bi
  XP           R$ 5 bi



CVM (10/2026): R$ 3,16 tri divulgados + R$ 769 bi confidenciais = R$ 3,92 tri
RMD 2.7 Fundos+Previdência (09/2026): R$ 4,10 tri → resíduo R$ 173 bi
  (o resíduo NÃO é 'a previdência fechada': mistura EFPCs, diferenças de
   perímetro e classificação; a PREVIC mede os títulos federais das EFPCs
   em R$ 921 bi — medida direta separada, não a diferença entre as fontes)


**Os nomes, resumidos:** os maiores detentores de títulos públicos entre os
fundos são **fundos de previdência e de renda fixa dos grandes bancos**:
BRASILPREV TOP TPF (R$ 274 bi), BB RF IV (R$ 154 bi), CAIXA PÓS FIXADO
PREVINVEST (R$ 105 bi), CFIN TÍTULOS PÚBLICOS PLENAS (R$ 82 bi). Por grupo:
**BB (R$ 680 bi), Caixa (R$ 419), BrasilPrev (R$ 394), Itaú (R$ 313),
Bradesco (R$ 207), Santander (R$ 148)** — os seis grupos somam 68,5% do total;
só os cinco bancos (sem a BrasilPrev) somam 56,0%.

Ou seja: o grupo "Fundos + Previdência" do RMD tem nomes — e são, em grande
medida, **veículos dos próprios bancos** (fundos próprios e de previdência
aberta), além dos fundos independentes. O look-through final (o cotista do
fundo) continua não público — mas o nível institucional já é nomeado, para os
fundos que divulgam carteira.


## Não-residentes, por país de domicílio (CVM)

A CVM compila a carteira declarada dos investidores não residentes por país
de domicílio tributário. É a resposta para "de onde vem o capital estrangeiro
que financia a dívida". Fonte: CVM, série histórica Investimento Estrangeiro —
carteira por país, **posição de dez/2024** (último dado da série histórica;
não é o mês corrente).


In [12]:
pais = pd.read_parquet(p / "dados" / "processado" / "cvm_carteira_pais.parquet")
print(f"títulos federais em não-residentes ({pais.mes.max():%m/%Y}): "
      f"{fmt_bi(pais.valor_bi.sum())}")
top = pais.nlargest(10, "valor_bi")
for _, r in top.iterrows():
    print(f"  {fmt_num(r.valor_bi):>8}  {r.pais[:38]}")
d27 = pd.read_parquet(p / "dados" / "processado" / "rmd_detentores.parquet")
dez24 = d27[d27.data == "2024-12-31"]
nr = dez24[dez24.detentor == "Não-residentes"].valor_bi.sum()
print(f"\nvalidação: CVM {fmt_bi(pais.valor_bi.sum())} vs RMD 2.7 "
      f"(não-residentes, dez/2024) {fmt_bi(nr)} — diferença conceitual: "
      "declaração dos representantes legais vs posição em custódia")


títulos federais em não-residentes (12/2024): R$ 607 bi
     196,2  ESTADOS UNIDOS DA AMÉRICA
     137,0  REINO UNIDO
      42,3  CINGAPURA
      32,3  CANADÁ
      30,4  LUXEMBURGO
      25,8  CHINA
      25,0  HOLANDA
      16,3  MÉXICO
      15,2  FRANÇA
      11,8  IRLANDA

validação: CVM R$ 607 bi vs RMD 2.7 (não-residentes, dez/2024) R$ 711 bi — diferença conceitual: declaração dos representantes legais vs posição em custódia


**Os países:** os **Estados Unidos (R$ 196 bi)** e o **Reino Unido (R$ 137 bi)**
respondem por mais de metade dos títulos federais em poder de não-residentes
(R$ 607 bi no total, dez/2024); depois vêm Cingapura (R$ 42 bi), Canadá
(R$ 32 bi), Luxemburgo (R$ 30 bi), China (R$ 26 bi) e Holanda (R$ 25 bi).
Os fundos domiciliados em Luxemburgo/Irlanda são, em boa parte, veículos de
investidores de outros países — o dado nomeia o domicílio fiscal, não o
beneficiário final (a mesma ressalva de sempre).


## Previdência fechada: o lado que faltava (PREVIC)

A previdência fechada (fundos de pensão patronais, as EFPCs) **não é fundo
CVM** — por isso não aparece no CDA. A PREVIC publica o Relatório da
Previdência Complementar Fechada, com a carteira de investimentos por
segmento. É a peça que completa o lado da previdência fechada.

Fonte: PREVIC, *Relatório 2025 da Previdência Complementar Fechada*
(jul/2026), Tabela 6 — posição de dez/2025.


In [13]:
# carteira de renda fixa das EFPCs, dez/2025 (R$ bi) — Tabela 6 do relatório
previc = pd.DataFrame([
    ("NTN-B e NTN-C — vencimento > 10 anos", 416.0),
    ("NTN-B e NTN-C — vencimento 5–10 anos", 186.9),
    ("NTN-B e NTN-C — vencimento < 5 anos", 171.2),
    ("Operações compromissadas", 130.8),
    ("LFT", 123.0),
    ("Títulos privados de emissão bancária", 51.6),
    ("Títulos privados de emissão não bancária", 24.4),
    ("NTN-F, LTN e outros", 24.1),
], columns=["segmento", "valor_bi"])
print(previc.assign(valor_bi=previc.valor_bi.map(fmt_num)).to_string(index=False))
tpf = previc[previc.segmento.str.contains("NTN|LFT|LTN")].valor_bi.sum()
print(f"\ntítulos federais das EFPCs (dez/2025): {fmt_bi(tpf)} — medida direta da PREVIC")
print("  não é o 'gap' CVM × RMD 2.7: aquele confronto (set–out/2026) deixa")
print(f"  {fmt_bi(resid)} depois de somar as carteiras confidenciais da CVM;")
print("  os R$ 921 bi das EFPCs são outra medida, de outro mês e outro perímetro.")


                                segmento valor_bi
    NTN-B e NTN-C — vencimento > 10 anos    416,0
    NTN-B e NTN-C — vencimento 5–10 anos    186,9
     NTN-B e NTN-C — vencimento < 5 anos    171,2
                Operações compromissadas    130,8
                                     LFT    123,0
    Títulos privados de emissão bancária     51,6
Títulos privados de emissão não bancária     24,4
                     NTN-F, LTN e outros     24,1

títulos federais das EFPCs (dez/2025): R$ 921 bi — medida direta da PREVIC
  não é o 'gap' CVM × RMD 2.7: aquele confronto (set–out/2026) deixa
  R$ 173 bi depois de somar as carteiras confidenciais da CVM;
  os R$ 921 bi das EFPCs são outra medida, de outro mês e outro perímetro.


**O que a PREVIC acrescenta:** a previdência fechada carrega **R$ 774 bi
em NTN-B e NTN-C** (títulos indexados à inflação, 69% da sua renda fixa) —
e o próprio relatório da PREVIC descreve a estratégia como **imunização de
longo prazo**, com concentração nos vencimentos acima de 10 anos (R$ 416 bi).

O quadro público dos detentores, nos três níveis, é este:
- **instituição**: BB, Caixa, BrasilPrev, Itaú, Bradesco, Santander (fundos
  que divulgam carteira);
- **país**: EUA (R$ 196 bi) e Reino Unido (R$ 137 bi) na frente;
- **risco**: a previdência carrega a inflação longa; bancos e fundos carregam
  a Selic de curto prazo; estrangeiros, o nominal.

O look-through final (o cotista do fundo, o sócio do plano) segue não público —
mas o nível institucional, que é o que move a política, está nomeado e
quantificado **para a parte que divulga**. A outra parte (R$ 769 bi em
carteiras confidenciais) é a lacuna declarada.


## Até onde o look-through público chega

A pergunta do dono final tem uma resposta estrutural: os dados públicos
identificam **o veículo** (o fundo, seu CNPJ e, quando divulga, a carteira) e
**o tipo** (fundo, EFPC, não-residente), mas não o **cotista final** nem a
carteira detalhada dos fundos com dispensa de divulgação. Os números abaixo
mostram onde a luz chega e onde ela para.


In [14]:
# fundos com "EXCLUSIVO" no nome (indício de fundo exclusivo): o nome costuma
# revelar o patrocinador/segmento, não o cotista final
blc = pd.read_csv(z.open("cda_fi_BLC_1_202610.csv"), sep=";", encoding="latin-1",
                  low_memory=False)
tp = blc[blc.TP_APLIC == "Títulos Públicos"].copy()
tp["VL"] = pd.to_numeric(tp.VL_MERC_POS_FINAL, errors="coerce") / 1e9
excl = tp[tp.DENOM_SOCIAL.str.contains("EXCLUSIVO|EXCLUSIVE", case=False, na=False)]
u = excl.groupby("DENOM_SOCIAL").VL.sum().sort_values(ascending=False)
print(f"nomes de fundos com 'EXCLUSIVO' e títulos públicos: {len(u)} nomes, "
      f"{fmt_bi(u.sum())}")
for nome, v in u.head(8).items():
    print(f"  {fmt_num(v):>8}  {nome[:62]}")
print("  (o nome do fundo costuma identificar o patrocinador/segmento: FAT = governo;")
print("   CAPEF = previdência de banco estatal; Itaú Private = private banking —")
print("   mas não prova quem é o cotista final)")


nomes de fundos com 'EXCLUSIVO' e títulos públicos: 40 nomes, R$ 82 bi
      42,1  BB EXTRAMERCADO EXCLUSIVO FAT FUNDO DE INVESTIMENTO FINANCEIRO
       9,0  CAIXA EXTRAMERCADO EXCLUSIVO FAT FUNDO DE INVESTIMENTO FINANCE
       4,2  CAIXA EXCLUSIVO XS4 FUNDO DE INVESTIMENTO FINANCEIRO RENDA FIX
       3,9  CAPEF EXCLUSIVO FUNDO DE INVESTIMENTO FINANCEIRO RENDA FIXA
       2,7  CAIXA EXTRAMERCADO EXCLUSIVO LOTERIAS FIF RENDA FIXA LONGO PRA
       2,7  CAPEF EXCLUSIVO II FUNDO DE INVESTIMENTO FINANCEIRO RENDA FIXA
       2,2  FUNDO DE INVESTIMENTO FINANCEIRO EXCLUSIVO MUNICH RE DO BRASIL
       1,7  LAWTON EXCLUSIVO FIF MULTIMERCADO CRÉDITO PRIVADO INVESTIMENTO
  (o nome do fundo costuma identificar o patrocinador/segmento: FAT = governo;
   CAPEF = previdência de banco estatal; Itaú Private = private banking —
   mas não prova quem é o cotista final)


In [15]:
# o MURO: os fundos com carteira confidencial (dispensa de divulgação)
tpc = conf[conf.TP_APLIC == "Títulos Públicos"]
total_divulgado = tp.VL.sum()
print(f"carteira CONFIDENCIAL (posição não divulgada): "
      f"{fmt_int(conf.CNPJ_FUNDO_CLASSE.nunique())} classes de fundos")
print(f"títulos públicos nessas carteiras: {fmt_bi(tpc.VL.sum())}")
print(f"  = {fmt_pct(tpc.VL.sum() / (total_divulgado + tpc.VL.sum()))} do total "
      "(divulgado + confidencial)")
print(f"  = {fmt_pct(tpc.VL.sum() / total_divulgado)} do que é divulgado")

# concentração pelo PL (cda_fi_PL): top 30 classes
pl = pd.read_csv(z.open("cda_fi_PL_202610.csv"), sep=";", encoding="latin-1",
                 low_memory=False)
pl["PL"] = pd.to_numeric(pl.VL_PATRIM_LIQ, errors="coerce")
top30 = pl.nlargest(30, "PL").PL.sum() / 1e12
total_pl = pl.PL.sum() / 1e12
print(f"\ntop 30 classes por PL: R$ {fmt_num(top30, 2)} tri "
      f"({fmt_pct(top30 / total_pl)} do PL total de R$ {fmt_num(total_pl, 2)} tri)")


carteira CONFIDENCIAL (posição não divulgada): 11.060 classes de fundos
títulos públicos nessas carteiras: R$ 769 bi
  = 19,6% do total (divulgado + confidencial)
  = 24,3% do que é divulgado



top 30 classes por PL: R$ 2,82 tri (19,7% do PL total de R$ 14,29 tri)


**Onde a luz chega:** o nome do fundo (quando divulga) costuma revelar o
patrocinador ou o segmento — FAT (governo), CAPEF (previdência de banco
estatal), Itaú Private (private banking), ApexBrasil (agência pública); o CNPJ
do fundo é público e leva ao administrador/gestor; e a previdência tem o
**patrocínio nomeado** pela PREVIC (BB, Caixa, Petrobras, governo federal…),
que é o look-through mais longo que existe legalmente. Isso identifica o
**veículo**, não necessariamente o cotista final.

**Onde a luz para:** R$ 769 bi em títulos públicos (**19,6%** do total
divulgado + confidencial; **24%** do que é divulgado) estão em **carteiras
confidenciais** — a CVM dispensa de divulgação os fundos exclusivos e de
cotistas restritos. É onde a identificação pública termina: a lei protege o nome,
e o dado público não ultrapassa esse muro. O que se pode dizer honestamente:
**há uma massa de capital que o registro público não nomeia — e ela é grande o
bastante para importar**.

A concentração, pelo que é visível, já é forte: 30 **classes** de fundos detêm
R$ 2,8 tri (19,7% do PL de todos os fundos); e os fundos com "EXCLUSIVO" no
nome somam R$ 82 bi em títulos federais (indício, não prova, de cotista único).


## A segunda frente: o spread do crédito

O sistema financeiro remunera o capital pelos **dois lados**: o Tesouro
paga sobre a dívida que ele detém (a seção acima) e os tomadores de crédito
pagam sobre o empréstimo que ele faz. A conta do spread fecha o elo entre
o P2 (quem recebe) e o P4 (i > g): a margem do crédito é a segunda fonte
de receita real alta.

Fontes: SGS 20740 (taxa média PF livres), 20786/20809 (spread do crédito
livre, total/PF), 20542/20570 (saldo da carteira livre), 21082/21084
(inadimplência). As séries 20783/20785 medem o spread do crédito **total**
(inclui o direcionado) e não são usadas nesta conta.


In [16]:
from src.extratores.base import ler_parquet
sg = ler_parquet("sgs")

# séries com tempestividades diferentes (a Selic segue adiante): alinhar
# pelo último mês em que TODAS as séries têm dado — sem isso a conta
# mistura meses
series = ["saldo_credito_livre", "saldo_credito_livre_pf", "spread_livre",
          "spread_livre_pf", "taxa_media_pf_livres", "selic_mensal_anualizada"]
ref = min(sg[sg.serie == s].data.max() for s in series)
def ult(nome):
    s = sg[(sg.serie == nome) & (sg.data == ref)]
    return float(s.valor.iloc[-1])
saldo, saldo_pf = ult("saldo_credito_livre") / 1000, ult("saldo_credito_livre_pf") / 1000
spread, spread_pf = ult("spread_livre"), ult("spread_livre_pf")
taxa_pf, selic = ult("taxa_media_pf_livres"), ult("selic_mensal_anualizada")
print(f"referência: {ref:%m/%Y} (último mês comum a todas as séries)")
print(f"carteira livre: {fmt_bi(saldo)} (PF: {fmt_bi(saldo_pf)})")
print(f"taxa média PF: {fmt_num(taxa_pf)}% | Selic: {fmt_num(selic, 2)}% | "
      f"spread livre PF: {fmt_num(spread_pf)} p.p.")
print("  (bases diferentes: a taxa média é das novas operações; o spread do")
print("   BCB é contra o custo de captação — não contra a Selic)")
print()
# o que cabe ao sistema financeiro da dívida (nossa estimativa por grupo)
jd = ind.juros_por_detentor(2025).groupby("detentor").juros_estimado_bi.sum()
sf = jd[["Instituições Financeiras", "Fundos de Investimento", "Seguradoras"]].sum()
if_bi = jd["Instituições Financeiras"]
print("as frentes da receita do sistema financeiro:")
print(f"  1. juros da dívida que cabem ao sistema financeiro (IF+Fundos+Seg): {fmt_bi(sf)}")
print(f"     (IF sozinho: {fmt_bi(if_bi)} por participação; R$ 285 bi por título —")
print("      mesma família RMD, não validação independente)")
print(f"  2. spread do crédito livre (RECEITA BRUTA): {fmt_bi(saldo * spread / 100)}  "
      f"(PF: {fmt_bi(saldo_pf * spread_pf / 100)})")
print()
print("o que esta conta NÃO é:")
print('  - "margem sobre a Selic": o funding dos bancos não é a Selic —')
print("    depósitos à vista rendem ~0; a conta não é receita de ninguém.")
print("  - spread menos inadimplência não é margem líquida: a inadimplência")
print("    (21084, PF total — inclui direcionado) é o ESTOQUE em atraso;")
print("    o custo de perda do ano é a despesa de provisão (outra grandeza,")
print("    não coletada). O spread é bruto: risco, operação e tributos dentro.")


referência: 09/2026 (último mês comum a todas as séries)
carteira livre: R$ 4,19 tri (PF: R$ 2,58 tri)
taxa média PF: 62,4% | Selic: 13,78% | spread livre PF: 48,5 p.p.
  (bases diferentes: a taxa média é das novas operações; o spread do
   BCB é contra o custo de captação — não contra a Selic)

as frentes da receita do sistema financeiro:
  1. juros da dívida que cabem ao sistema financeiro (IF+Fundos+Seg): R$ 503 bi
     (IF sozinho: R$ 289 bi por participação; R$ 285 bi por título —
      mesma família RMD, não validação independente)
  2. spread do crédito livre (RECEITA BRUTA): R$ 1,49 tri  (PF: R$ 1,25 tri)

o que esta conta NÃO é:
  - "margem sobre a Selic": o funding dos bancos não é a Selic —
    depósitos à vista rendem ~0; a conta não é receita de ninguém.
  - spread menos inadimplência não é margem líquida: a inadimplência
    (21084, PF total — inclui direcionado) é o ESTOQUE em atraso;
    o custo de perda do ano é a despesa de provisão (outra grandeza,
    não coletada).

In [17]:
t = sg[sg.serie.isin(["taxa_media_pf_livres", "selic_mensal_anualizada",
                       "spread_livre_pf"])].copy()
t = t.pivot_table(index="data", columns="serie", values="valor")
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(t.index, t.taxa_media_pf_livres, color=VERMELHO, lw=1.8,
        label="taxa média de juros — PF, crédito livre (% a.a.)")
ax.plot(t.index, t.selic_mensal_anualizada, color=AZUL, lw=1.8,
        label="Selic mensal anualizada (% a.a.)")
ax.plot(t.index, t.spread_livre_pf, color=VERDE, lw=1.8,
        label="spread do crédito livre — PF (p.p.)")
ax.set_ylabel("% a.a. (taxas) / p.p. (spread)")
ax.set_title("A segunda frente: taxa de crédito PF, Selic e spread do crédito livre")
ax.legend(fontsize=9, frameon=False)
salvar(fig, "02_spread_credito")


fig -> viz/figs/02_spread_credito.png


PosixPath('/home/vanessa/devel/juros_e_divida/viz/figs/02_spread_credito.png')

**O que a conta mostra:** a receita bruta de spread do crédito livre é
**cerca de três vezes** os juros da dívida que cabem ao sistema financeiro:
~R$ 1,49 tri/ano de spread bruto (PF: R$ 1,25 tri) contra ~R$ 503 bi/ano para
IF+fundos+seguradoras. O sistema financeiro remunera o capital pelas duas
portas: o Tesouro paga sobre a dívida que ele detém; os tomadores de crédito
pagam sobre o empréstimo que ele faz.

**O que a conta NÃO mostra (e está declarado na célula acima):** o spread
é **bruto** — não é lucro; a margem sobre a Selic não é receita de ninguém
(o funding bancário é mais barato que a Selic); e a inadimplência não é o
custo de perda do ano (é o estoque em atraso, e da carteira PF total). O
que falta para a conta do líquido: a **despesa de provisão** (o custo de
risco real) e o **ROE** (o retorno do capital) — pendências declaradas.


**Achados verificáveis (P2).**

- O **sistema financeiro** — Instituições Financeiras + Fundos de Investimento —
  detém cerca de metade do estoque (54,2% em set/2026) e é o maior beneficiário
  estimado do fluxo de juros (só as instituições financeiras: ~R$ 290 bi no ano
  em 2025; por título, R$ 285 bi).
- **Fundos de Investimento** (~22,9%) e **Previdência** (~22,4%) vêm em seguida,
  praticamente empatados; a previdência inclui fundos de pensão patronais,
  planos de servidores e trabalhadores privados.
- **Pessoas físicas** via Tesouro Direto têm participação pequena: ~3% do
  estoque — o Tesouro Direto direto, sem contar a exposição via fundos e
  previdência. O padrão de beneficiário é **institucional**, não pulverizado
  como o discurso do "Tesouro Direto democratizou" sugere.
- **Não-residentes** oscilaram entre ~5% e ~19% nas posições de dezembro da
  série completa (2007–2025); desde 2016, ficaram em geral entre ~9% e ~14% —
  parcela estruturalmente estrangeira, sensível a crises.

**Limitação:** saber *quais* instituições, dentro de cada grupo, detêm quanto
não é publicável (sigilo; Req. 696/2019 negado). O que é público é o nível
institucional dos fundos que divulgam carteira; o cotista final e a carteira
detalhada dos fundos com dispensa continuam lacuna informada, não escondida.
